
# Credit Card Fraud Detection using Deep Learning

**Author:** Mihir  
**Project Type:** End‑to‑End Deep Learning Portfolio Project  

This notebook demonstrates:

- Data cleaning
- Exploratory Data Analysis
- Feature engineering
- Deep learning model building
- Model training
- Model evaluation

Dataset: Kaggle Credit Card Fraud Detection Dataset



## Step 1 — Environment Setup
Install required libraries if not already installed.


In [ ]:

# Uncomment if running in a fresh environment
# !pip install pandas numpy matplotlib seaborn scikit-learn tensorflow streamlit joblib


## Step 2 — Imports

In [ ]:

import os
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

warnings.filterwarnings("ignore")


## Step 3 — Load Dataset

In [ ]:

file_path = "creditcard.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully")
else:
    raise FileNotFoundError("Download creditcard.csv from Kaggle and place it in this folder")

df.head()


In [ ]:

print("Dataset shape:", df.shape)
df.info()
print(df.isnull().sum())


## Step 4 — Data Cleaning

In [ ]:

duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates()

print("Dataset shape after cleaning:", df.shape)


In [ ]:

print(df.describe())


In [ ]:

plt.figure(figsize=(6,4))
sns.histplot(df['Amount'], bins=50, kde=True)
plt.title("Transaction Amount Distribution")
plt.show()


## Step 5 — Exploratory Data Analysis

In [ ]:

plt.figure(figsize=(6,4))
sns.countplot(x='Class', data=df)
plt.title("Fraud vs Normal Transactions")
plt.show()


In [ ]:

plt.figure(figsize=(8,5))
sns.boxplot(x='Class', y='Amount', data=df)
plt.title("Transaction Amount by Class")
plt.show()


In [ ]:

plt.figure(figsize=(8,4))
sns.histplot(df['Time'], bins=50, kde=True)
plt.title("Transaction Time Distribution")
plt.show()


In [ ]:

plt.figure(figsize=(12,10))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()


## Step 6 — Feature Engineering

In [ ]:

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train = y_train.to_numpy()
y_test = y_test.to_numpy()


In [ ]:

classes = np.array([0,1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print("Class weights:", class_weights)


## Step 7 — Deep Learning Model

In [ ]:

input_dim = X_train_scaled.shape[1]

model = Sequential()

model.add(Input(shape=(input_dim,)))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


## Step 8 — Model Training

In [ ]:

EPOCHS = 20
BATCH_SIZE = 64

history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    shuffle=True
)


In [ ]:

plt.figure()
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Training vs Validation Loss")
plt.legend(["train","validation"])
plt.show()


In [ ]:

plt.figure()
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Training vs Validation Accuracy")
plt.legend(["train","validation"])
plt.show()


## Step 9 — Model Evaluation

In [ ]:

y_pred_prob = model.predict(X_test_scaled).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


In [ ]:

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix")
print(cm)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()


In [ ]:

print(classification_report(y_test, y_pred))
